In [ ]:
import pickle
from typing import Literal
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root
import tqdm
import tqdm.auto
tqdm.auto.tqdm = tqdm.tqdm
import pandas as pd

from python.config import load_config

cfg = load_config(os.environ.get('CONFIG', 'config.json'))

import python.plots as plots
from python.error_metrics import LoadedErrorMetrics, FilteredErrorMetrics, MetricsIndex as MI # Alias
from python.sampled_hierarchies import SampledHierarchies, SingleStepHierarchies
from python.statistical_test import slope_ttest, page_trend_test, run_comprehensive_per_series_tests, PerSeriesTrendTestResultStore
from python.aggregation import Aggregation

from itertools import product

In [ ]:
AVAILABLE_LEVELS = Aggregation.complete_summing_matrix(n=cfg.LEVELS.SUBSET, seed=42)

if cfg.is_single_step_hierarchies():
    sh = SingleStepHierarchies(AVAILABLE_LEVELS)
else:
    sh = SampledHierarchies(AVAILABLE_LEVELS, cfg.LEVELS.DIFF, cfg.LEVELS.SEQUENCES, max_h=cfg.LEVELS.MAX_H)

datasets = ['web-traffic', 'M4', 'M5']
models = ['AutoETS', 'AutoARIMA', 'Prophet']

In [ ]:
results = {}

test_fun = page_trend_test if cfg.TEST == 'page l' else slope_ttest

for d, m in tqdm.tqdm(product(datasets, models)):
    ID = f"{d}/{m}"
    tqdm.tqdm.write(ID)
    em = FilteredErrorMetrics(LoadedErrorMetrics(f"metrics/{cfg.MASTER_FOLDER}/{ID}/sample_hiers_errors.csv"), MI(error_metric='nrmse'))
    results[ID] = run_comprehensive_per_series_tests(sh, em, 'nrmse')

with open(f"metrics/{cfg.MASTER_FOLDER}/per_series_test_results.pkl", 'wb') as f:
    pickle.dump(results, f)


In [ ]:
with open(f"metrics/{cfg.MASTER_FOLDER}/per_series_test_results.pkl", 'rb') as f:
    results: dict[str, PerSeriesTrendTestResultStore] = pickle.load(f)

In [ ]:
def extract_results(dataset: str, model: str,
                    granularity: Literal['top', 'bottom'],
                    trend: Literal['increasing', 'decreasing', 'stagnant'],
                    test_type: Literal['page l', 'slope t', 'sign', 'wincoxon'] | None = None):
    g = getattr(results[f"{dataset}/{model}"], granularity)
    t = getattr(g, trend)
    if test_type:
        return t[test_type]
    return t

In [ ]:
g = 'top'
t = 'increasing'
test = 'wilcoxon'

df = pd.DataFrame({ m: { d: extract_results(d, m, g, t, test).fraction_significant
               for d in datasets }
               for m in models })
print(df)

In [ ]:
model = cfg.MODEL
dataset = cfg.DATASET.NAME

plots.plot_per_series_pvalues(extract_results(dataset, model, g, t, test), bins=50)

In [ ]:
rows = []
for d in datasets:
    for m in models:
        for g in ['top', 'bottom']:
            for t in ['increasing', 'decreasing', 'stagnant']:
                for test in ['page l', 'slope t', 'sign', 'wilcoxon']:
                    test_dict = extract_results(d, m, g, t)
                    if test in test_dict:
                        r = test_dict[test]
                        rows.append({
                            'dataset': d,
                            'model': m,
                            'granularity': g,
                            'trend': t,
                            'test_method': test,
                            'fraction_significant': r.fraction_significant,
                            'aggregate_p_value': r.p_value
                        })

df = pd.DataFrame(rows).set_index(['dataset', 'model', 'granularity', 'test_method']).pivot(columns=['trend'])
df['increasing<50'] = df[('fraction_significant','increasing')] < 0.5
df['insignificant%'] = 1.0 - df[('fraction_significant','increasing')] - df[('fraction_significant','decreasing')] - df[('fraction_significant','stagnant')]
df